# ASR architecture benchmark on TED-LIUM 3 (Colab T4)

This is the architecture comparison task: Whisper, Wav2Vec2 CTC, FastConformer CTC, FastConformer RNNT, Paraformer, and Qwen3-ASR. It uses a separate candidate registry and Drive run root from the earlier Whisper-heavy diagnostic. Run cells top to bottom. The real 6-model probe must pass before validation; the native-test run is available only after the validation selection receipt is complete. Prior native-test exposure means test results remain diagnostic.

The six systems differ in training data, parameter count, and toolchain as well as architecture; do not interpret score differences as architecture-only causal effects.

Paraformer is decoded with the pinned checkpoint tokens.json. Earlier runs using its mismatched SentencePiece ID map are invalid for A4. A changed bundle SHA selects a fresh Drive run root; do not mix results from different roots.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, pathlib, shutil, subprocess, sys, zipfile
DRIVE = pathlib.Path('/content/drive/MyDrive/asr_architecture_colab_t4')
BUNDLE = DRIVE / 'asr_architecture_colab_t4_inputs_empty_transcript_261005_clean.zip'
assert BUNDLE.is_file() and BUNDLE.with_suffix(BUNDLE.suffix + '.sha256').is_file(), 'Upload the architecture bundle and its .sha256 file to the Drive folder.'
expected = BUNDLE.with_suffix(BUNDLE.suffix + '.sha256').read_text().split()[0]
LOCAL = pathlib.Path('/content/asr_architecture') / expected[:12]
REPO = LOCAL / 'multimodal-lecture-summarizer'
RUN_ROOT = DRIVE / 'runs' / f'architecture_{expected[:12]}'
local_zip = pathlib.Path('/content') / BUNDLE.name
shutil.copy2(BUNDLE, local_zip)
actual = hashlib.sha256(local_zip.read_bytes()).hexdigest()
assert actual == expected, 'bundle SHA-256 mismatch'
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_zip) as archive:
    for member in archive.infolist():
        target = (REPO / member.filename).resolve()
        assert target.is_relative_to(REPO.resolve()), f'unsafe archive path: {member.filename}'
    archive.extractall(REPO)
assert (REPO / 'experiments/scripts/asr_architecture_colab_run.py').is_file()
HF_HOME = DRIVE / 'model_cache' / 'huggingface'
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['HF_HUB_OFFLINE'] = '0'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Verified bundle:', actual, 'repo:', REPO, 'run root:', RUN_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verified bundle: 25ce9f1f300bd1944f081aac9c1543ee6fe3d2a027105c7f03f637505051027b repo: /content/asr_architecture/25ce9f1f300b/multimodal-lecture-summarizer run root: /content/drive/MyDrive/asr_architecture_colab_t4/runs/architecture_25ce9f1f300b


In [2]:
# Build isolated Python 3.11 workers from the checked-in pin file. Stop on resolver or pip-check errors.
import yaml
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
LOCK = yaml.safe_load((REPO / 'experiments/registry/asr_architecture_environment.lock.yaml').read_text())
ENV_ROOT = pathlib.Path('/content/asr-architecture-envs')
ENV_ROOT.mkdir(exist_ok=True)
def make_env(name):
    pins = LOCK['groups'][name]
    env_dir = ENV_ROOT / name
    py = env_dir / 'bin/python'
    if not py.is_file():
        subprocess.run(['uv', 'venv', '--python', LOCK['python'], '--seed', str(env_dir)], check=True)
    subprocess.run(['uv', 'pip', 'install', '--python', str(py), f"torch=={pins['torch']}", f"torchaudio=={pins['torchaudio']}", '--index-url', LOCK['cuda_wheel_indexes'][name]], check=True)
    packages = [f'{pkg}=={version}' for pkg, version in pins.items() if pkg not in {'torch','torchaudio'}]
    subprocess.run(['uv', 'pip', 'install', '--python', str(py), *packages], check=True)
    subprocess.run([str(py), '-m', 'pip', 'check'], check=True)
    return str(py)
PY_BASE, PY_NEMO, PY_FUNASR = make_env('base'), make_env('nemo'), make_env('funasr')
# CTranslate2 4.4.0 expects cuDNN 8; keep it separate from PyTorch's cuDNN 9.
CUDNN8 = pathlib.Path('/content/asr-architecture-cudnn8')
if not (CUDNN8 / 'nvidia/cudnn/lib').is_dir():
    subprocess.run([PY_BASE, '-m', 'pip', 'install', '--target', str(CUDNN8), 'nvidia-cudnn-cu12==8.9.7.29'], check=True)
site_packages = pathlib.Path(subprocess.check_output([PY_BASE, '-c', 'import site; print(site.getsitepackages()[0])'], text=True).strip())
cuda_libs = [str(CUDNN8 / 'nvidia/cudnn/lib')] + [str(site_packages / 'nvidia' / name / 'lib') for name in ('cublas', 'cudnn') if (site_packages / 'nvidia' / name / 'lib').is_dir()]
os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(cuda_libs + ([os.environ['LD_LIBRARY_PATH']] if os.environ.get('LD_LIBRARY_PATH') else []))
os.environ.update(ASR_ARCH_PYTHON=PY_BASE, ASR_ARCH_PYTHON_BASE=PY_BASE, ASR_ARCH_PYTHON_NEMO=PY_NEMO, ASR_ARCH_PYTHON_FUNASR=PY_FUNASR)
print('Worker interpreters:', {k:v for k,v in os.environ.items() if k.startswith('ASR_ARCH_PYTHON')})


Worker interpreters: {'ASR_ARCH_PYTHON': '/content/asr-architecture-envs/base/bin/python', 'ASR_ARCH_PYTHON_BASE': '/content/asr-architecture-envs/base/bin/python', 'ASR_ARCH_PYTHON_NEMO': '/content/asr-architecture-envs/nemo/bin/python', 'ASR_ARCH_PYTHON_FUNASR': '/content/asr-architecture-envs/funasr/bin/python'}


In [3]:
# Download only the six immutable model snapshots declared in the architecture registry.
import yaml
registry = yaml.safe_load((REPO / 'experiments/registry/asr_architecture_candidates.yaml').read_text())
assert len(registry['candidates']) == 6
script = 'from huggingface_hub import snapshot_download; import sys; print(snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2]))'
for candidate in registry['candidates']:
    assert len(candidate['revision']) == 40, candidate['id']
    subprocess.run([PY_BASE, '-c', script, candidate['model_repo'], candidate['revision']], check=True, env=os.environ.copy())
print('All exact revisions cached:', [c['revision'] for c in registry['candidates']])

All exact revisions cached: ['ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66', 'b91e708eeb1a28886fdcf9906f8a7c49d0baae24', '330c1b6694fed5ae3797bdbca754a315fa382df7', '3a4e74f2585fc5b1258d58eed640faf5322423ed', 'd043d60434a65ae49b2b2462a09660049871b8e9', 'c4468bdb552ddc559e464f6081e22dd4034f2e68']


In [4]:
# Preflight plus one real validation audio inference for every family. Full work stays closed if any row fails.
os.environ['MLS_ASR_CANDIDATE_SET'] = 'architecture'
subprocess.run([PY_BASE, '-m', 'experiments.scripts.asr_architecture_preflight', '--output', str(RUN_ROOT / 'preflight.json'), '--drive-root', str(DRIVE)], cwd=REPO, check=True, env=os.environ.copy())
probe = subprocess.run([PY_BASE, '-m', 'experiments.scripts.asr_architecture_colab_run', 'probe', '--drive-root', str(RUN_ROOT)], cwd=REPO, check=False, env=os.environ.copy())
receipt = RUN_ROOT / 'architecture_probe.json'
if receipt.is_file():
    for row in json.loads(receipt.read_text())['outcomes']:
        print(row['candidate'], row['status'], row.get('reason', ''), flush=True)
if probe.returncode:
    raise RuntimeError(f'Architecture probe failed ({probe.returncode}); full benchmark is blocked: {receipt}')

RQ0_ARCH.A0_WhisperBase passed 
RQ0_ARCH.A1_Wav2Vec2CTC passed 
RQ0_ARCH.A2_FastConformerCTC passed 
RQ0_ARCH.A3_FastConformerRNNT passed 
RQ0_ARCH.A4_ParaformerEN passed 
RQ0_ARCH.A5_Qwen3ASR_0_6B passed 


In [8]:
# Validation selection: six candidates × 507 frozen validation units.
subprocess.run([PY_BASE, '-m', 'experiments.scripts.asr_architecture_colab_run', 'validation', '--drive-root', str(RUN_ROOT)], cwd=REPO, check=True, env=os.environ.copy())
print('Validation selection receipt:', RUN_ROOT / 'validation/diagnostic/full/selection_receipt.json')

Validation selection receipt: /content/drive/MyDrive/asr_architecture_colab_t4/runs/architecture_25ce9f1f300b/validation/diagnostic/full/selection_receipt.json


In [9]:
# Diagnostic native test: six candidates × 1,155 units. Earlier test access prevents confirmatory claims.
subprocess.run([PY_BASE, '-m', 'experiments.scripts.asr_architecture_colab_run', 'native_test', '--drive-root', str(RUN_ROOT)], cwd=REPO, check=True, env=os.environ.copy())
print('Parent index:', RUN_ROOT / 'native_test/diagnostic/full/parent_index.json')

Parent index: /content/drive/MyDrive/asr_architecture_colab_t4/runs/architecture_25ce9f1f300b/native_test/diagnostic/full/parent_index.json
